In [1]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))

if project_root not in sys.path:
    sys.path.append(project_root)

In [2]:
import pipeline.src.python.config as cfg
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)

In [3]:
import networkx as nx

In [4]:
MAGAZINE_1 = 'scopus'
DATASET_TEXT_FEATURE = (
    "text"  # In the dataset file, the column name that contains the text data
)
cfg_dict_1 = cfg.MAGAZINE_CONFIG[MAGAZINE_1]

In [9]:
MAGAZINE_2 = 'the_guardian'
cfg_dict_2 = cfg.MAGAZINE_CONFIG[MAGAZINE_2]

In [10]:
MAGAZINE_3 = 'science_news'
cfg_dict_3 = cfg.MAGAZINE_CONFIG[MAGAZINE_3]

In [11]:
from bertopic import BERTopic

model_path = cfg.MODELS_FOLDER / f'{MAGAZINE_1}/model_0.343.safetensors'
model_1 = BERTopic.load(model_path, 
                      embedding_model=cfg.EMBEDDING_MODEL
                      )

/home/banfi/.uve/cuda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
model_path_2 = cfg.MODELS_FOLDER / f'{MAGAZINE_2}/model_0.334.safetensors'

model_2 = BERTopic.load(model_path_2,
                        cfg.EMBEDDING_MODEL
                        )


In [13]:
model_path_3 = cfg.MODELS_FOLDER / f'{MAGAZINE_3}/model_0.309.safetensors'

model_3 = BERTopic.load(model_path_3,
                        cfg.EMBEDDING_MODEL
                        )


In [15]:
data = np.load(cfg_dict_1['OUTPUT_PATH'],allow_pickle=True) 

ids = data['id']
texts = data['text'] 
embeddings = data['embedding'] 
documents = data['clean_text']

In [16]:
data_2 = np.load('/home/banfi/TETYS/pipeline/src/python/data/interim/embeddings/the_guardian/the_guardian_embeddings_summary_filtered.npz',allow_pickle=True) 

ids_2 = data_2['id']
texts_2 = data_2['text'] 
embeddings_2 = data_2['embedding'] 
documents_2 = data_2['clean_text']

In [17]:
data_3 = np.load(cfg_dict_3['OUTPUT_PATH'],allow_pickle=True) 

ids_3 = data_3['id']
texts_3 = data_3['text'] 
embeddings_3 = data_3['embedding'] 
documents_3 = data_3['clean_text']

### Model 1 vs Model 2

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_2 = BTM(model_1=model_1,
                            model_2=model_2,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_2,
                            embeddings_2=embeddings_2,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_2.title())

2026-02-09 14:09:37,164 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:37,370 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [25]:
models_metrics_1_vs_2.evaluate_metrics()

In [26]:
model_1_vs_model_2_closeness = models_metrics_1_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [ ]:
inverse_models_metrics_2_vs_1 = BTM(model_1=model_2,
                                    model_2=model_1,
                                    ids_1=ids_2,
                                    texts_1=texts_2,
                                    embeddings_1=embeddings_2,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_2.title(),model_2_name=MAGAZINE_1.title())

2026-02-09 14:09:40,871 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:42,092 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [28]:
inverse_models_metrics_2_vs_1.evaluate_metrics()

In [29]:
model_2_vs_model_1_closeness = inverse_models_metrics_2_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Model 1 vs Model 3

In [ ]:
from pipeline.src.python.btm import BTM

models_metrics_1_vs_3 = BTM(model_1=model_1,
                            model_2=model_3,
                            ids_1=ids,
                            texts_1=texts,
                            embeddings_1=embeddings,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_1.title(),
                            model_2_name=MAGAZINE_3.title())

2026-02-09 14:09:54,709 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:09:54,721 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [31]:
models_metrics_1_vs_3.evaluate_metrics()

In [32]:
model_1_vs_model_3_closeness = models_metrics_1_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [ ]:
inverse_models_metrics_3_vs_1 = BTM(model_1=model_3,
                                    model_2=model_1,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts,
                                    embeddings_2=embeddings,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_1.title())

2026-02-09 14:11:39,363 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:11:40,577 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.38246655464172363


In [34]:
inverse_models_metrics_3_vs_1.evaluate_metrics()

In [35]:
model_3_vs_model_1_closeness = inverse_models_metrics_3_vs_1.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

### Model 2 vs Model 3

In [36]:
from pipeline.src.python.btm import BTM

models_metrics_2_vs_3 = BTM(model_1=model_2,
                            model_2=model_3,
                            ids_1=ids_2,
                            texts_1=texts_2,
                            embeddings_1=embeddings_2,
                            texts_2=texts_3,
                            embeddings_2=embeddings_3,
                            model_1_name=MAGAZINE_2.title(),
                            model_2_name=MAGAZINE_3.title()
                            )

2026-02-09 14:14:37,245 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:14:37,257 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3790895342826843


In [37]:
models_metrics_2_vs_3.evaluate_metrics()

In [38]:
model_2_vs_model_3_closeness = models_metrics_2_vs_3.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

In [39]:
inverse_models_metrics_3_vs_2 = BTM(model_1=model_3,
                                    model_2=model_2,
                                    ids_1=ids_3,
                                    texts_1=texts_3,
                                    embeddings_1=embeddings_3,
                                    texts_2=texts_2,
                                    embeddings_2=embeddings_2,
                                    model_1_name=MAGAZINE_3.title(),
                                    model_2_name=MAGAZINE_2.title())

2026-02-09 14:16:10,197 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.
2026-02-09 14:16:10,400 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Threshold:0.3820500373840332


In [41]:
inverse_models_metrics_3_vs_2.evaluate_metrics()

In [40]:
model_3_vs_model_2_closeness = inverse_models_metrics_3_vs_2.get_topic_closeness().sort_values(by='Topic Closeness',ascending=False)

## Renaming

In [46]:
model_1_vs_model_2_closeness = model_1_vs_model_2_closeness.rename(columns={'Scopus Topic Label':'Model 1 Label',
                                                                            'The_Guardian Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [47]:
model_2_vs_model_1_closeness = model_2_vs_model_1_closeness.rename(columns={'Scopus Topic Label':'Model 2 Label',
                                                                            'The_Guardian Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [49]:
model_1_vs_model_3_closeness = model_1_vs_model_3_closeness.rename(columns={'Scopus Topic Label':'Model 1 Label',
                                                                            'Science_News Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [50]:
model_3_vs_model_1_closeness = model_3_vs_model_1_closeness.rename(columns={'Scopus Topic Label':'Model 2 Label',
                                                                            'Science_News Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [52]:
model_2_vs_model_3_closeness = model_2_vs_model_3_closeness.rename(columns={'The_Guardian Topic Label':'Model 1 Label',
                                                                            'Science_News Topic Label':'Model 2 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

In [53]:
model_3_vs_model_2_closeness = model_3_vs_model_2_closeness.rename(columns={'The_Guardian Topic Label':'Model 2 Label',
                                                                            'Science_News Topic Label':'Model 1 Label'})[['Model 1 Label','Model 2 Label','Topic Closeness']]

### Creates dataset

In [58]:
closeness = pd.concat([model_1_vs_model_2_closeness,
                model_2_vs_model_1_closeness,
                model_1_vs_model_3_closeness,
                model_3_vs_model_1_closeness,
                model_2_vs_model_3_closeness,
                model_3_vs_model_2_closeness])


In [63]:
closeness.to_parquet('edges.parquet')

In [26]:
counts = model_1.get_topic_freq()['Count'][1:].to_list() if -1 in model_1.topics_ else model_1.get_topic_freq()['Count'].to_list()
labels  = model_1.custom_labels_[1:] if -1 in model_1.topics_ else model_1.custom_labels_

model_1_nodes = pd.DataFrame({'Topic Label': labels,
                              'Degree':counts })
model_1_nodes['Model'] = MAGAZINE_1.title()


In [28]:
counts_2 = model_2.get_topic_freq()['Count'][1:].to_list() if -1 in model_2.topics_ else model_2.get_topic_freq()['Count'].to_list()
labels_2  = model_2.custom_labels_[1:] if -1 in model_2.topics_ else model_2.custom_labels_

model_2_nodes = pd.DataFrame({'Topic Label': labels_2,
                              'Degree':counts_2 })
model_2_nodes['Model'] = MAGAZINE_2.title()

In [30]:
counts_3 = model_3.get_topic_freq()['Count'][1:].to_list() if -1 in model_3.topics_ else model_3.get_topic_freq()['Count'].to_list()
labels_3  = model_3.custom_labels_[1:] if -1 in model_3.topics_ else model_3.custom_labels_

model_3_nodes = pd.DataFrame({'Topic Label': labels_3,
                              'Degree':counts_3 })
model_3_nodes['Model'] = MAGAZINE_3.title()

In [32]:
nodes = pd.concat([model_1_nodes,
                model_2_nodes,
                model_3_nodes])

In [33]:
nodes.to_parquet('nodes.parquet')

## Retriving data

In [119]:
import pandas as pd
edges = pd.read_parquet('edges.parquet')
nodes = pd.read_parquet('nodes.parquet')

### Vedere come riuscire a eliminare i bridges ( alta closeness basso alignment)

In [120]:
topics_1 = []
topics_2 = []
closeness = []

for _, edge in edges.iterrows():

    topic1 = edge['Model 1 Label']
    topic2 = edge['Model 2 Label']
    c = edge['Topic Closeness']

    bidirection = (
        (edges['Model 2 Label'] == topic1) &
        (edges['Model 1 Label'] == topic2)
    )

    if bidirection.any():

        topics_1.append(topic1)
        topics_2.append(topic2)
        closeness.append(c)


In [121]:
results = pd.DataFrame({'Model 1 Label': topics_1,
                        'Model 2 Label': topics_2,
                        'Topic Closeness':closeness})

In [122]:
results = results.sort_values(by=['Model 1 Label','Topic Closeness'],ascending=[False,False])

In [123]:
len(results)

1982

In [124]:
results_to_be_filtered = results.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')

In [126]:
results_to_be_filtered['N_Match'] = results_to_be_filtered['Topic Closeness']  * results_to_be_filtered['Degree']

In [127]:
results_to_be_filtered = results_to_be_filtered[ results_to_be_filtered['N_Match'] > 5 ]

In [128]:
topic_list = results_to_be_filtered['Model 1 Label'].unique()


In [129]:
unique_topic = list(topic_list)

In [130]:
topic_dict = { topic:0 for topic in unique_topic}

In [131]:
topics_1_final = []
topics_2_final = []
closeness_final = []

for _, edge in results_to_be_filtered.iterrows():

    topic1 = edge['Model 1 Label']
    topic2 = edge['Model 2 Label']
    c = edge['Topic Closeness']

    insert = False
    if c > 0.1:
        insert = True
    #elif topic_dict[topic1] <= 3 and c > 0.05:
    #    insert = True
    
    if insert == True:
        topics_1_final.append(topic1)
        topics_2_final.append(topic2)
        closeness_final.append(c)

In [132]:
final_results  = pd.DataFrame({'Model 1 Label': topics_1_final,
                        'Model 2 Label': topics_2_final,
                        'Topic Closeness':closeness_final})

In [133]:
edges_filtered = final_results.copy()

## Start

In [382]:
edges_to_be_filtered = edges.merge(nodes,how='inner',left_on='Model 1 Label',right_on='Topic Label')

In [383]:
# Min SN Cluster 10
FIXED_THRESHOLD_SN = 5

# Min TG Cluster 50
FIXED_THRESHOLD_TG = 15

# Min Scopus Cluster 80
FIXED_THRESHOLD_SCOPUS = 15

# Percentage
PERCENTAGE_THRESHOLD = 0.05

In [384]:
common_topics = pd.read_csv('common_topics.csv',sep=";", encoding='utf-8')

In [385]:
edges_to_be_filtered['N_Match'] = edges_to_be_filtered['Topic Closeness']  * edges_to_be_filtered['Degree']

In [386]:
edges_sn_filtered = edges_to_be_filtered[
     (edges_to_be_filtered['Model'] == 'Science_News') &
     (edges_to_be_filtered['N_Match'] > FIXED_THRESHOLD_SN) &
     (edges_to_be_filtered['Topic Closeness'] > PERCENTAGE_THRESHOLD)# &
     #(edges_to_be_filtered['Model 1 Label'].isin(common_topics['Labels'])) &
     #(edges_to_be_filtered['Model 2 Label'].isin(common_topics['Labels']))
     ]

In [387]:
len(edges_sn_filtered)

64

In [388]:
edges_scopus_filtered = edges_to_be_filtered[ (edges_to_be_filtered['Model'] == 'Scopus') &
                                             (edges_to_be_filtered['N_Match'] > FIXED_THRESHOLD_SCOPUS) &
                                             (edges_to_be_filtered['Topic Closeness'] > PERCENTAGE_THRESHOLD) #&
                                            #(edges_to_be_filtered['Model 1 Label'].isin(common_topics['Labels'])) &
                                            #(edges_to_be_filtered['Model 2 Label'].isin(common_topics['Labels']))
                                             ]

In [389]:
len(edges_scopus_filtered)

573

In [390]:
edges_tg_filtered = edges_to_be_filtered[ (edges_to_be_filtered['Model'] == 'The_Guardian') &
                                        (edges_to_be_filtered['N_Match'] > FIXED_THRESHOLD_TG) &
                                        (edges_to_be_filtered['Topic Closeness'] > PERCENTAGE_THRESHOLD) #&
                                       # (edges_to_be_filtered['Model 1 Label'].isin(common_topics['Labels'])) &
                                       # (edges_to_be_filtered['Model 2 Label'].isin(common_topics['Labels']))
                                           ]

In [295]:
len(edges_tg_filtered)

240

In [296]:
edges_filtered = pd.concat([edges_sn_filtered,edges_scopus_filtered,edges_tg_filtered])

In [297]:
edges_filtered = edges_filtered[['Model 1 Label','Model 2 Label','Topic Closeness']]

## end

In [134]:

G = nx.DiGraph(description='Connected components') 

for _, node in nodes.iterrows():
    G.add_node(
    node['Topic Label'],
    degree=node['Degree'],
    model=node['Model']
)

for _, edge in edges_filtered.iterrows():
    G.add_edge(
    edge['Model 1 Label'],
    edge['Model 2 Label'],
    weight=edge['Topic Closeness']
)



## Min

In [135]:
import networkx as nx

# raccogli i pesi direzionali
dir_weights = {}

for u, v, data in G.edges(data=True):
    w = data["weight"]
    dir_weights[(u, v)] = w

In [136]:
G_und = nx.Graph()

# costruisci solo se esistono entrambe le direzioni
for (u, v), w_uv in dir_weights.items():

    if (v, u) in dir_weights:
        w_vu = dir_weights[(v, u)]

        w = min(w_uv, w_vu)

        # evita doppioni
        if not G_und.has_edge(u, v):
            G_und.add_edge(u, v, weight=w)


## Simmetric mean

In [270]:
def harmonic_mean(a, b, eps=1e-9):
    return 2*a*b/(a+b+eps)

G_und = nx.Graph()

for (u, v), w_uv in dir_weights.items():

    if (v, u) in dir_weights:
        w_vu = dir_weights[(v, u)]

        w = harmonic_mean(w_uv, w_vu)

        if not G_und.has_edge(u, v):
            G_und.add_edge(u, v, weight=w)


## Louvain

In [271]:
import community as community_louvain

In [272]:
partition = community_louvain.best_partition(G_und)

In [273]:
distinct_partition = []
n_partition = [ distinct_partition.append(v) for k,v in partition.items() if v not in distinct_partition]

In [274]:
len(distinct_partition)

40

In [275]:
from collections import defaultdict

clusters = defaultdict(list)

for node, comm in partition.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 0:
Tuberculosis Resistance and Control, Tuberculosis Treatment Challenges

Cluster 10:
Zoonotic Disease Surveillance, Disease Spread and Environmental Impact, WNV Surveillance and Detection, Zika Outbreak and Mosquito Spread, Mpox Outbreak Transmission, Mpox Outbreak in Africa, Rabies Prevention and Control, Zika Virus Outbreaks, Zika Virus Outbreak Risk, Zika Virus and Microcephaly, SARS-CoV-2 Animal Infection Transmission, Climate and Health Impact, Heatwave Mortality Risk, Travel Medicine and Infectious Disease Surveillance, Disease Outbreak Tracking Initiative

Cluster 2:
Infection Control and Prevention, Hospital Infection Control Improvement, CDI Incidence and Prevention, Hospital Surface Sterilization and Contamination Control, MRSA Hospital Surveillance, Neonatal MRSA Outbreak

Cluster 3:
Salmonella Resistance and Prevalence, Salmonella Food Contamination Outbreak, Campylobacter Infections and Prevalence, Shiga Toxin–Producing E. coli O157 Infections, Listeria Outbreak 

In [237]:
label_name = []
cluster_number = []
for k,v in partition.items():
    label_name.append(k)
    cluster_number.append(v)

In [196]:
louvain_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

In [197]:
louvain_cluster.to_csv('louvain_new_approach.csv',sep='ç',encoding='utf-8')

## Leiden cluster creation

In [312]:
import leidenalg
import igraph as ig

In [313]:
nodes = list(G_und.nodes())
G_ig = ig.Graph.from_networkx(G_und)
G_ig.vs["name"] = nodes

In [314]:
partition = leidenalg.find_partition(
    G_ig,
    leidenalg.ModularityVertexPartition,
    weights="weight",
    n_iterations=-1
)

In [315]:
clusters = partition.membership


In [316]:
node_to_cluster = {
    v["name"]: clusters[i]
    for i, v in enumerate(G_ig.vs)
}


In [317]:
from collections import defaultdict

clusters = defaultdict(list)

for node, comm in node_to_cluster.items():
    clusters[comm].append(node)

for c, nodes in clusters.items():
    print(f"Cluster {c}:")
    print(", ".join(nodes))
    print()


Cluster 24:
Tuberculosis Resistance and Control, Tuberculosis Treatment Challenges

Cluster 0:
Zoonotic Disease Surveillance, Disease Spread and Environmental Impact, WNV Surveillance and Detection, Zika Outbreak and Mosquito Spread, Zika Virus Outbreaks, Zika Virus Outbreak Risk, Zika Virus and Microcephaly, Climate and Health Impact, Disease Outbreak Tracking Initiative

Cluster 10:
Infection Control and Prevention, Hospital Infection Control Improvement, Hospital Surface Sterilization and Contamination Control, MRSA Hospital Surveillance

Cluster 1:
SARS-CoV-2 Genomic Evolution, Covid-19 Pandemic Mitigation Strategies, Coronavirus Death Trends in England, COVID-19 Spread and Mortality, Global Coronavirus Outbreak Death Toll, School Outbreak Transmission, School Reopening and Educational Needs, Coronavirus Death Toll Surpasses Million

Cluster 4:
HPAI Outbreak and Pathogenicity, Flu Virus Spread Patterns, H5N1 Bird Flu Outbreak in Poultry Industry, Viral Replication in Influenza Viru

In [318]:
label_name = []
cluster_number = []
for k,v in node_to_cluster.items():
    label_name.append(k)
    cluster_number.append(v)


In [319]:
sorted(cluster_number)[-1]

44

In [320]:
leiden_cluster = pd.DataFrame({'Topic Label':label_name,
              'Cluster': cluster_number})

In [321]:
i = 0
leiden_cluster.to_csv(f'snapshot_{i}.csv',sep='ç',encoding='utf-8')

## Start loop

In [419]:
leiden_cluster = leiden_cluster.merge(leiden_cluster['Cluster'].value_counts().reset_index().rename(columns={'count':'Cluster size'}),on='Cluster')

In [420]:
unclustered_topics = edges[ ~ edges['Model 1 Label'].isin(leiden_cluster['Topic Label'])]

In [421]:
unclustered_topics_with_closeness = unclustered_topics.merge(leiden_cluster,how='left',left_on='Model 2 Label',right_on='Topic Label')

### Removing empty clusters

In [422]:
unclustered_topics_with_closeness = unclustered_topics_with_closeness[ ~ unclustered_topics_with_closeness['Topic Label'].isna()]

### Somma

In [248]:
unclustered_topics_with_closeness_aggregated = unclustered_topics_with_closeness.groupby(['Model 1 Label','Cluster','Cluster size']).agg(Topic_Closeness_Sum=('Topic Closeness','sum')).reset_index()

In [250]:
unclustered_topics_with_closeness_aggregated['Topic Closeness Mean'] = unclustered_topics_with_closeness_aggregated['Topic_Closeness_Sum'] / unclustered_topics_with_closeness_aggregated['Cluster size']

### Alternativa

In [423]:
k = 3
top_3_closeness = ( 
    unclustered_topics_with_closeness
    .groupby(['Model 1 Label','Cluster','Cluster size'],as_index=False)
    .head(k) 
    .groupby(['Model 1 Label','Cluster','Cluster size'])['Topic Closeness']
    .agg(Topic_Closeness_Sum="sum")
    )

In [424]:
top_3_closeness = top_3_closeness.reset_index()

In [425]:
top_3_closeness['Topic Closeness Mean'] = top_3_closeness['Topic_Closeness_Sum'] / k

In [426]:
top_3_closeness[ top_3_closeness['Model 1 Label'] == 'Pandemic Impact and Hope' ]

,Model 1 Label,Cluster,Cluster size,Topic_Closeness_Sum,Topic Closeness Mean
1395,Pandemic Impact and Hope,0.0,12.0,0.454545,0.151515
1396,Pandemic Impact and Hope,6.0,6.0,0.090909,0.030303
1397,Pandemic Impact and Hope,8.0,4.0,0.181818,0.060606
1398,Pandemic Impact and Hope,9.0,7.0,0.181818,0.060606
1399,Pandemic Impact and Hope,20.0,6.0,0.090909,0.030303
1400,Pandemic Impact and Hope,24.0,2.0,0.181818,0.060606


In [427]:
unclustered_topics_with_closeness_aggregated = top_3_closeness.copy()

### Nuovo fine

In [428]:
unclustered_topics_with_closeness_aggregated[ unclustered_topics_with_closeness_aggregated['Model 1 Label'] == 'Pandemic Impact and Hope' ]

,Model 1 Label,Cluster,Cluster size,Topic_Closeness_Sum,Topic Closeness Mean
1395,Pandemic Impact and Hope,0.0,12.0,0.454545,0.151515
1396,Pandemic Impact and Hope,6.0,6.0,0.090909,0.030303
1397,Pandemic Impact and Hope,8.0,4.0,0.181818,0.060606
1398,Pandemic Impact and Hope,9.0,7.0,0.181818,0.060606
1399,Pandemic Impact and Hope,20.0,6.0,0.090909,0.030303
1400,Pandemic Impact and Hope,24.0,2.0,0.181818,0.060606


In [429]:
unclustered_topics_with_closeness_aggregated.sort_values(by=['Topic Closeness Mean'],ascending=[False])

,Model 1 Label,Cluster,Cluster size,Topic_Closeness_Sum,Topic Closeness Mean
698,Genomic Surveillance for AMR Pathogens,2.0,15.0,1.380000,0.460000
899,Household Transmission of Cov-2,1.0,19.0,1.237500,0.412500
691,Genetic diversity and reservoir role of bat-associated alpha- and betacoronaviruses.,6.0,6.0,1.218391,0.406130
599,European Vaccine Supply Challenges,3.0,11.0,1.102273,0.367424
1171,MERS Infection in Camels and Humans,6.0,6.0,1.066176,0.355392
...,...,...,...,...,...
1787,Salmonella Resistance and Prevalence,5.0,7.0,0.000406,0.000135
1794,Salmonella Resistance and Prevalence,27.0,2.0,0.000406,0.000135
882,High Sensitivity Biosensor for Bacterial Cell Detection,10.0,4.0,0.000208,0.000069
883,High Sensitivity Biosensor for Bacterial Cell Detection,11.0,4.0,0.000208,0.000069


In [430]:
top_3_unclustered_topics = unclustered_topics_with_closeness_aggregated.groupby('Model 1 Label',group_keys=False).apply(lambda x: x.nlargest(3,"Topic Closeness Mean")).reset_index()

/tmp/ipykernel_415086/1299132218.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_3_unclustered_topics = unclustered_topics_with_closeness_aggregated.groupby('Model 1 Label',group_keys=False).apply(lambda x: x.nlargest(3,"Topic Closeness Mean")).reset_index()


In [431]:
top_1_unclustered_topics = unclustered_topics_with_closeness_aggregated.groupby('Model 1 Label',group_keys=False).apply(lambda x: x.nlargest(1,"Topic Closeness Mean")).reset_index()

/tmp/ipykernel_415086/1823038874.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  top_1_unclustered_topics = unclustered_topics_with_closeness_aggregated.groupby('Model 1 Label',group_keys=False).apply(lambda x: x.nlargest(1,"Topic Closeness Mean")).reset_index()


In [432]:
top_1_unclustered_topics = top_1_unclustered_topics.astype({'Cluster':"int64","Cluster size":"int64"})

In [433]:
top_1_unclustered_topics.sort_values(by=['Topic Closeness Mean'],ascending=[False]).head(20)

,index,Model 1 Label,Cluster,Cluster size,Topic_Closeness_Sum,Topic Closeness Mean
132,698,Genomic Surveillance for AMR Pathogens,2,15,1.380000,0.460000
168,899,Household Transmission of Cov-2,1,19,1.237500,0.412500
130,691,Genetic diversity and reservoir role of bat-associated alpha- and betacoronaviruses.,6,6,1.218391,0.406130
110,599,European Vaccine Supply Challenges,3,11,1.102273,0.367424
213,1171,MERS Infection in Camels and Humans,6,6,1.066176,0.355392
150,796,HIV and COVID-19 Interaction,11,4,1.016667,0.338889
194,1059,Intranasal SARS-CoV-2 Vaccine Platform,3,11,0.941919,0.313973
370,2120,Wastewater Surveillance for Cov-2 Detection,0,12,0.914544,0.304848
309,1764,SARS-CoV-2 Spike and Entry Mechanisms,1,19,0.832117,0.277372
356,2056,Vaccine Autism Controversy,3,11,0.773481,0.257827


In [434]:
added_topics = top_1_unclustered_topics[ top_1_unclustered_topics['Topic Closeness Mean'] >= 0.10][['Model 1 Label','Cluster','Cluster size']]

In [435]:
added_topics = added_topics.rename(columns={'Model 1 Label':'Topic Label'})

In [436]:
len(added_topics)

67

In [437]:
len(leiden_cluster)

184

In [438]:
leiden_cluster = pd.concat([leiden_cluster,added_topics.head(10)])

In [439]:
leiden_cluster = leiden_cluster.drop(columns=['Cluster size'])

In [440]:
i += 1
leiden_cluster.to_csv(f'snapshot_{i}.csv',sep='ç',encoding='utf-8')

In [441]:
i

5